In [4]:
import json
import numpy as np
import faiss
import re

from sentence_transformers import SentenceTransformer

Load embedding model

In [5]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Load knowledge base

In [6]:
kb_path = r"C:\Users\Moetez\OneDrive\ml_tutor_project\data\ml_knowledge_base\ml_concepts.json"

with open(kb_path, "r", encoding="utf-8") as f:

    knowledge_base = json.load(f)

documents = [
    item["content"]
    for item in knowledge_base
]

Create knowledge embeddings

In [7]:
doc_embeddings = embedding_model.encode(documents)

Build FAISS Index

In [8]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(doc_embeddings))

print("Knowledge Base Size:", index.ntotal)

Knowledge Base Size: 4


Retrieval function

In [9]:
def retrieve_knowledge(query, top_k=2):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append(knowledge_base[idx])

    return results

Topic extraction

In [10]:
def extract_topic(conversation_history):

    topics = [
        "overfitting",
        "gradient descent",
        "linear regression",
        "decision trees",
        "regularization",
        "neural networks"
    ]

    text = " ".join(conversation_history).lower()

    for topic in topics:

        if topic in text:
            return topic

    return "machine learning"

Smart question rewriter

In [11]:
def rewrite_question(conversation_history, question):

    topic = extract_topic(conversation_history)

    q = question.lower()

    if "why" in q:

        return f"Why is {topic} important in machine learning?"

    elif "how" in q:

        return f"How does {topic} work in machine learning?"

    elif "explain" in q:

        return f"Can you explain {topic} simply?"

    elif "prevent" in q:

        return f"How can overfitting be prevented?"

    else:

        return f"{question} about {topic}"

Confusion detector

In [12]:
def detect_confusion(question):

    q = question.lower()

    patterns = [
        r"don't understand",
        r"confused",
        r"not clear",
        r"explain again",
        r"simplify",
        r"what do you mean",
        r"hard to understand"
    ]

    for pattern in patterns:

        if re.search(pattern, q):

            return True

    return False

Adaptive teaching styles

In [13]:
teaching_styles = {

    "overfitting": {

        "beginner":
        "Overfitting happens when a model memorizes training data instead of learning general patterns.",

        "analogy":
        "Imagine memorizing exam answers without understanding the subject. That is similar to overfitting.",

        "concise":
        "Overfitting reduces model generalization.",

        "detailed":
        "Overfitting occurs when a model learns noise and details from training data, causing poor performance on unseen examples."
    },

    "gradient descent": {

        "beginner":
        "Gradient descent improves machine learning models step by step by reducing errors.",

        "analogy":
        "Imagine walking downhill in fog while searching for the lowest point. That is similar to gradient descent.",

        "concise":
        "Gradient descent minimizes loss iteratively.",

        "detailed":
        "Gradient descent updates model parameters iteratively to minimize prediction error."
    }
}

Memory retrieval

In [14]:
def retrieve_relevant_memory(
    conversation_history,
    query,
    top_k=3
):

    embeddings = embedding_model.encode(
        conversation_history
    )

    memory_index = faiss.IndexFlatL2(
        embeddings.shape[1]
    )

    memory_index.add(np.array(embeddings))

    query_embedding = embedding_model.encode([query])

    distances, indices = memory_index.search(
        np.array(query_embedding),
        top_k
    )

    relevant_memory = []

    for idx in indices[0]:

        relevant_memory.append(
            conversation_history[idx]
        )

    return relevant_memory

Main tutor pipeline

In [15]:
def conversational_ml_tutor(
    conversation_history,
    current_question
):

    # Step 1 — Rewrite Question
    rewritten_question = rewrite_question(
        conversation_history,
        current_question
    )

    # Step 2 — Retrieve Relevant Memory
    relevant_memory = retrieve_relevant_memory(
        conversation_history,
        rewritten_question
    )

    # Step 3 — Retrieve Knowledge
    retrieved_docs = retrieve_knowledge(
        rewritten_question
    )

    # Step 4 — Detect Confusion
    confused = detect_confusion(
        current_question
    )

    # Step 5 — Topic Detection
    topic = extract_topic(
        conversation_history
    )

    # Step 6 — Select Teaching Mode
    if confused:

        mode = "analogy"

    elif "briefly" in current_question.lower():

        mode = "concise"

    elif "detail" in current_question.lower():

        mode = "detailed"

    else:

        mode = "beginner"

    # Step 7 — Generate Response
    if topic in teaching_styles:

        response = teaching_styles[topic][mode]

    else:

        response = retrieved_docs[0]["content"]

    return {

        "rewritten_question": rewritten_question,

        "relevant_memory": relevant_memory,

        "retrieved_knowledge": retrieved_docs,

        "confusion_detected": confused,

        "teaching_mode": mode,

        "response": response
    }

Test tutor

In [16]:
conversation_history = [

    "What is overfitting?",

    "Overfitting happens when a model memorizes training data.",

    "Why is it bad?"
]

question = "I still don't understand."

Run tutor

In [17]:
result = conversational_ml_tutor(
    conversation_history,
    question
)

result

{'rewritten_question': "I still don't understand. about overfitting",
 'relevant_memory': ['What is overfitting?',
  'Overfitting happens when a model memorizes training data.',
  'Why is it bad?'],
 'retrieved_knowledge': [{'topic': 'overfitting',
   'difficulty': 'beginner',
   'content': 'Overfitting happens when a machine learning model memorizes training data too closely and performs poorly on unseen data.'},
  {'topic': 'gradient descent',
   'difficulty': 'beginner',
   'content': "Gradient descent is an optimization algorithm used to minimize a model's error by adjusting parameters step by step."}],
 'confusion_detected': True,
 'teaching_mode': 'analogy',
 'response': 'Imagine memorizing exam answers without understanding the subject. That is similar to overfitting.'}

In [18]:
print("="*80)

print("REWRITTEN QUESTION:")
print(result["rewritten_question"])

print("\nRELEVANT MEMORY:")
print(result["relevant_memory"])

print("\nCONFUSION DETECTED:")
print(result["confusion_detected"])

print("\nTEACHING MODE:")
print(result["teaching_mode"])

print("\nFINAL RESPONSE:")
print(result["response"])

REWRITTEN QUESTION:
I still don't understand. about overfitting

RELEVANT MEMORY:
['What is overfitting?', 'Overfitting happens when a model memorizes training data.', 'Why is it bad?']

CONFUSION DETECTED:
True

TEACHING MODE:
analogy

FINAL RESPONSE:
Imagine memorizing exam answers without understanding the subject. That is similar to overfitting.
